# Quickstart: load a design, simulate, synthesize, and read the results

This notebook runs the FIFO example of Xeda. It simulates the design, synthesizes it, reads the
results and the files of a run, and runs the same request again to see that Xeda reuses the run.

The notebook needs Verilator, Yosys and `nextpnr-ecp5` on your `PATH`. It writes only under
`notebook-output/` in its working directory.

## Set up

A `Workspace` holds the options of a launch. The most important option is the run root: the
directory where Xeda writes every run directory. Xeda resolves the path once, when it creates the
workspace.

In [ ]:
import os
from pathlib import Path

import xeda
from xeda import Design, Spec, Workspace

# The examples of the Xeda checkout. Set XEDA_EXAMPLES to use another copy.
EXAMPLES = Path(os.environ.get("XEDA_EXAMPLES", Path.cwd().parent / "examples")).resolve()
if not EXAMPLES.is_dir():
    raise FileNotFoundError(
        f"No examples directory at {EXAMPLES}. "
        "Set the XEDA_EXAMPLES environment variable to the `examples` directory of an Xeda checkout."
    )

# This notebook writes only under WORK.
WORK = Path("notebook-output").resolve()
ws = Workspace(WORK / "xeda_run")
print("Xeda", xeda.__version__)
print("run root:", ws.run_root)

## Load a design

`Design.load` reads a design file and checks it. It starts no tool. A `Design` does not change after
it is loaded: to get another design, make a copy with `with_updates`.

In [ ]:
design = Design.load(EXAMPLES / "sv/fifo/fifo.xeda.yaml")

print(design.name, "| top:", design.rtl.top)
print("rtl sources:", [source.path.name for source in design.rtl.sources])
print("tb sources: ", [source.path.name for source in design.tb.sources])
print("targets:", design.targets)

## Simulate

A `Spec` says what to run: a flow name, a design and explicit settings. `ws.run` runs it and
returns a `Result`. A tool that fails is an outcome in the result, not an exception.
`result.passed` answers the question "did it work?".

In [ ]:
sim = ws.run(Spec("verilator", design))
print(sim.outcome, sim.passed)

`result.record` is the record of the flow that you asked for. A simulation passes only on evidence
that it ended. Xeda keeps that evidence as metrics under `sim`. Every metric has a description, a
type and, for a physical quantity, a unit.

In [ ]:
record = sim.record
print(record.flow, record.outcome, f"{record.runtime:.1f} s")
print("ended by:", record.metric("sim.ended_by"))
print("errors:", record.metric("sim.errors"), "warnings:", record.metric("sim.warnings"))
print("time:", record.metric("sim.time"), record.metric_info["sim.time"].unit)

## Synthesize a chain

A spec can name a chain of flows with `+`. Each flow feeds the next one. Settings are scoped to a
flow: the key is the flow name. Dotted keys and nested tables mean the same. A clock period is a
physical quantity, so it is text with a unit. The settings that flows share, such as `fpga` and
`clock`, flow along the chain: set them once, on the first flow.

Here Yosys synthesizes the FIFO for a Lattice ECP5 part, and nextpnr places and routes it.

In [ ]:
synth = Spec(
    "yosys_fpga+nextpnr",
    design,
    settings={
        "yosys_fpga": {"fpga.part": "LFE5U-85F-6BG381C", "clock.period": "10 ns"},
        "nextpnr": {"lpf_allow_unconstrained": True},
    },
)
result = ws.run(synth)
print(result.outcome, result.passed)

`result.nodes` has one entry for every flow of the chain, in run order. `result.record` is the
record of the last flow, here `nextpnr`.

In [ ]:
for name, node in result.nodes.items():
    print(f"{name:11} {node.outcome.value:7} {node.elapsed:5.1f} s")

## Read the metrics

A metric has a dotted path. `record.metric(path)` reads it. `record.metric_info` describes every
metric that the run reports, with its unit.

In [ ]:
placed = result.record
for path in ("fmax", "wns", "timing_met", "lut", "ff"):
    info = placed.metric_info[path]
    print(f"{path:10} {placed.metric(path)!s:>8} {info.unit or '':4} {info.description[:60]}")

The two flows of the chain have their own records. Yosys counts the cells of the netlist by type.
nextpnr counts the used resources after placement. A path that a run does not report raises
`KeyError`.

In [ ]:
yosys = result.nodes["yosys_fpga"].record
print(dict(yosys.metric("cell_types")))

## Open the files of a run

A record stores file names relative to its run directory. To get the paths, open the record. While
the `with` block runs, no other launch can replace the files. A flow lists its outputs by name (a
file that another flow can read) and its artifacts by key (a file for you to read).

In [ ]:
with yosys.open() as run:
    print("outputs:  ", sorted(run.outputs))
    print("artifacts:", sorted(run.artifacts))
    netlist = run.output("netlist")
    print(netlist.name, netlist.stat().st_size, "bytes")
    print(len(run.artifacts["log"].read_text().splitlines()), "lines in the log of Yosys")

## Run it again

Nothing changed, so Xeda reuses the finished runs. The `disposition` of a node says if it ran
(`ran`) or was reused (`reused`).

In [ ]:
again = ws.run(synth)
print(*(node.disposition.value for node in again.nodes.values()))

## Next

- A launch that does not pass, a time limit and a cancel: `02-simulation.ipynb`.
- The record of a run is the document `results.json`. The record, the error codes and the
  command line are in the `machine-readable` page of the documentation.